In [1]:
import feedparser
import pandas as pd
import boto3
import json
from botocore.exceptions import ClientError
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv
from datetime import datetime
from bs4 import BeautifulSoup
import numpy as np
import time
import requests
from boto3.dynamodb.conditions import Key, Attr

In [ ]:
"""
# これは実行しなくてもOK。過去データを引っ張ってくるだけだから
# 一括書き込み。batch_writerを使用すると、DynamoDBの制限に基づいて自動的にバッチを分割してくれる。
def get_dynamo_data(days=20, table_name='ai_news'):
    ut = time.time()
    START_TIME = int(ut - days*24*60*60) # 7日間の範囲指定
    END_TIME = int(ut) #現在の時間

    client = boto3.client('dynamodb', region_name='ap-northeast-1')
    dynamodb = boto3.resource('dynamodb', region_name='ap-northeast-1')
    table = dynamodb.Table(table_name)

    response = table.query(
        IndexName = "published_datetime",
        KeyConditionExpression=(
            Key("category").eq("AI_news") &
            Key("published_datetime").between(START_TIME, END_TIME)
        )
    )

    items = response.get('Items', [])
    return items

news_data_json = get_dynamo_data(days=20, table_name='ai_news')
news_data_df = pd.DataFrame(news_data_json)
news_data_df.to_csv('news_data_0404_original.csv', index=False)

"""

In [2]:
# 作成者で作った正解データ
df_correct = pd.read_csv('news_data_0404.csv')
df_news_data = df_correct[["ttl","published_datetime","link","category","summary","title","id"]]
news_data_json = df_news_data.to_dict(orient='records')

In [3]:
from botocore.config import Config

def load_api_key():
    load_dotenv(dotenv_path="../.env", override=True)
    os.environ["AWS_BEARER_TOKEN_BEDROCK"] = os.getenv("Bedrock_API_Key")

def create_response(prompt_text,news_data):

    custom_config = Config(
        read_timeout=300,
        connect_timeout=60,
        retries={'max_attempts': 2, 'mode': 'standard'}
    )

    client = boto3.client("bedrock-runtime", config=custom_config)

    prompt = PromptTemplate(
        input_variables=["news_data"],
        template = prompt_text)
    prompt = prompt.format(news_data=news_data)

    body = json.dumps({
        "messages":[
            {"role":"user","content":prompt}
        ],
        "max_tokens":5000,
        "temperature":0.3,
        "anthropic_version":"bedrock-2023-05-31"
    })

    model_id = "anthropic.claude-3-5-sonnet-20240620-v1:0"

    response = client.invoke_model(
        modelId=model_id,
        body=body
    )

    return response

In [12]:
load_api_key()

def chunked_news_data(news_data, batch_size=10):
    for i in range(0, len(news_data), batch_size):
        yield news_data[i:i + batch_size]

f = open("prompt_eval.txt","r",encoding="utf-8")
prompt_text = f.read()

batch_results = []
for batch_index, batch in enumerate(chunked_news_data(news_data_json, 10), start=1):
    response = create_response(prompt_text, batch)
    response_body = json.loads(response.get("body").read())
    answer = response_body["content"][0]["text"]
    batch_output = json.loads(answer)
    batch_results.extend(batch_output)
    print(f"batch {batch_index}: {len(batch)} items processed")
print(f"total items from LLM: {len(batch_results)}")

parsed_answer = batch_results  # バッチ結果を直接使用
df_answer = pd.DataFrame(parsed_answer)

df = df_correct.merge(df_answer , on="link",how="left")
if df["priority"].isnull().sum() > 0:
    print("PriorityのカラムでNullが存在しています。確認してください")
else:
    print("priorityのカラムでNullはありません")
df.head(2)

batch 1: 10 items processed
batch 2: 10 items processed
batch 3: 10 items processed
batch 4: 10 items processed
batch 5: 10 items processed
batch 6: 10 items processed
batch 7: 10 items processed
batch 8: 10 items processed
total items from LLM: 80
priorityのカラムでNullはありません


,ttl,published_datetime,link,category,summary,title,id,correct_priority,priority
0,1775319484,1774109884,https://zenn.dev/karaage0703/articles/fcca40c6...,AI_news,NVIDIA DGX Spark（GB10、ARM64、128GB統合メモリ）でローカルLL...,DGX Sparkで色々なローカルLLMを動かした比較結果,202603221001,Medium,High
1,1775343600,1774134000,https://atmarkit.itmedia.co.jp/ait/articles/26...,AI_news,システムがますます企業収益に密接に関連するようになる中で、「オブザーバビリティー」（可観測性...,システム障害で「1分500万円」が消える、だが“監視だけ”ではビジネスを守れない？,202603221201,Low,Low


In [13]:
parsed_answer = batch_results  # バッチ結果を直接使用
df_answer = pd.DataFrame(parsed_answer)

df = df_correct.merge(df_answer , on="link",how="left")
if df["priority"].isnull().sum() > 0:
    print("PriorityのカラムでNullが存在しています。確認してください")
else:
    print("priorityのカラムでNullはありません")
df.head(2)

priorityのカラムでNullはありません


,ttl,published_datetime,link,category,summary,title,id,correct_priority,priority
0,1775319484,1774109884,https://zenn.dev/karaage0703/articles/fcca40c6...,AI_news,NVIDIA DGX Spark（GB10、ARM64、128GB統合メモリ）でローカルLL...,DGX Sparkで色々なローカルLLMを動かした比較結果,202603221001,Medium,High
1,1775343600,1774134000,https://atmarkit.itmedia.co.jp/ait/articles/26...,AI_news,システムがますます企業収益に密接に関連するようになる中で、「オブザーバビリティー」（可観測性...,システム障害で「1分500万円」が消える、だが“監視だけ”ではビジネスを守れない？,202603221201,Low,Low


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

# NaN値を削除してから型を統一
df_clean = df.dropna(subset=['correct_priority', 'priority'])
y_true = df_clean['correct_priority'].astype(str)    # 正解データ列
y_pred = df_clean['priority'].astype(str) # 予測データ列

labels = ['Low', 'Medium', 'High']
report_dict = classification_report(y_true, y_pred, labels=labels, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

print("### Classification Report ###")
report_df

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f'Actual_{l}' for l in labels], 
                         columns=[f'Pred_{l}' for l in labels])
print("\n### Confusion Matrix ###")
cm_df


### Classification Report ###

### Confusion Matrix ###


,Pred_Low,Pred_Medium,Pred_High
Actual_Low,22,18,9
Actual_Medium,1,7,3
Actual_High,0,3,17


In [15]:
report_df.to_clipboard()

In [16]:
cm_df.to_clipboard()

In [11]:
df["priority"].to_clipboard(index=False, header=False)